## Uses Derby as data catalog
### Derby is embedded, inprocess database, if it is loaded into spark on one notebook, it cannot be used on other spark notebook

### You need to stop the spark session in order to use this DB in write mode in another spark session

### ensure  to stop the spark.stop() or stop the worker and master and start master and worker again. 

In [ ]:
import os
import socket
from pyspark.sql import SparkSession

master_url = os.environ.get("SPARK_MASTER", f"spark://{socket.gethostname()}:7077")
warehouse = os.environ.get("SPARK_SQL_WAREHOUSE", "hdfs:///user/hive/warehouse")

catalog_dir = os.path.abspath(os.path.expanduser(
    os.environ.get("SPARK_CATALOG_DIR", "~/.spark-catalog/d32_olist_metastore")
))

# os.makedirs(catalog_dir, exist_ok=True)
catalog_url = f"jdbc:derby:;databaseName={catalog_dir};create=true"

spark = (
    SparkSession.builder
    .appName("D32-Olist-Spark-SQL-Lab")
    .master(master_url)
    .config("spark.sql.warehouse.dir", warehouse)
    .config("spark.sql.catalogImplementation", "hive")
    .config("javax.jdo.option.ConnectionURL", catalog_url)
    .config("hive.metastore.schema.verification", "false")
    .config("datanucleus.schema.autoCreateAll", "true")
    .enableHiveSupport()
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
assert spark.version.startswith("3.5.9"), f"This lab targets Spark 3.5.9; found {spark.version}"
print("Catalog:", spark.conf.get("spark.sql.catalogImplementation"))
print("Catalog metadata:", catalog_dir)
print("Warehouse:", spark.conf.get("spark.sql.warehouse.dir"))

In [ ]:
# Create and select the database
spark.sql("CREATE DATABASE IF NOT EXISTS orderdb")
spark.sql("USE orderdb")

In [ ]:
spark.sql("SHOW TABLES in orderdb").show()

In [ ]:
spark.sql("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id   INT,
    customer_name STRING,
    city          STRING
)
USING PARQUET
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS products (
    product_id   INT,
    product_name STRING,
    category     STRING,
    price        DECIMAL(10,2)
)
USING PARQUET
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS orders (
    order_id    INT,
    customer_id INT,
    order_date  DATE,
    status      STRING
)
USING PARQUET
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS order_items (
    order_id   INT,
    product_id INT,
    quantity   INT
)
USING PARQUET
""")

In [ ]:
spark.sql("SHOW TABLES").show()

In [ ]:
spark.sql("""
INSERT INTO customers VALUES
    (1, 'Arun',  'Bengaluru'),
    (2, 'Divya', 'Hyderabad'),
    (3, 'Kiran', 'Chennai'),
    (4, 'Meena', 'Pune')
""")


In [ ]:
spark.sql("""
SELECT * FROM customers 
""").show()

In [ ]:
spark.stop()